# Decision Trees — Demo

Walks through both classification and regression trees end-to-end on self-contained sklearn datasets.

**Pipeline:**
1. Classification — `load_breast_cancer()`: fit, visualise, evaluate
2. Show overfitting (no max_depth) vs pre-pruning (max_depth=3)
3. Post-pruning — sweep `ccp_alpha`
4. Feature importance
5. Regression — `load_diabetes()`: fit and evaluate

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, mean_squared_error, r2_score

np.random.seed(42)

## 1 — Classification Tree on Breast Cancer Data

30 numerical features, binary target (malignant / benign).

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names
target_names  = data.target_names

print('Shape:', X.shape)
print('Classes:', target_names)
print('Class distribution:', np.bincount(y))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 2 — Unbounded Tree (Shows Overfitting)

Without `max_depth`, the tree grows until every leaf is pure.

In [ ]:
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)

print(f'Depth:  {tree_full.get_depth()}')
print(f'Leaves: {tree_full.get_n_leaves()}')
print(f'Train accuracy: {tree_full.score(X_train, y_train):.3f}')
print(f'Test  accuracy: {tree_full.score(X_test,  y_test):.3f}')

Train accuracy = 1.0 → tree has memorised the training data. Test accuracy is lower → classic overfitting symptom.

## 3 — Pre-Pruning (max_depth=3)

In [ ]:
tree_small = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_small.fit(X_train, y_train)

print(f'Depth:  {tree_small.get_depth()}')
print(f'Leaves: {tree_small.get_n_leaves()}')
print(f'Train accuracy: {tree_small.score(X_train, y_train):.3f}')
print(f'Test  accuracy: {tree_small.score(X_test,  y_test):.3f}')

In [ ]:
# Visualise the small tree
plt.figure(figsize=(16, 8))
plot_tree(tree_small,
          feature_names=feature_names,
          class_names=target_names,
          filled=True, rounded=True, fontsize=8)
plt.show()

## 4 — Sweep max_depth (Train vs Test Curve)

The classic bias-variance plot for trees: train accuracy keeps rising, test accuracy peaks and then drops.

In [ ]:
rows = []
for d in [1, 2, 3, 5, 7, 10, 15, 20, None]:
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    rows.append({
        'max_depth': d if d is not None else 'unbounded',
        'leaves':    m.get_n_leaves(),
        'train_acc': round(m.score(X_train, y_train), 3),
        'test_acc':  round(m.score(X_test,  y_test),  3)
    })
print(pd.DataFrame(rows))

## 5 — Hyperparameter Tuning with GridSearchCV

In [ ]:
param_grid = {
    'max_depth':        [3, 5, 7, 10, None],
    'min_samples_split':[2, 10, 20],
    'min_samples_leaf': [1, 5, 10]
}

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid, cv=5, scoring='accuracy'
)
grid.fit(X_train, y_train)

print('Best params:', grid.best_params_)
print(f'Best CV accuracy: {grid.best_score_:.3f}')
print(f'Test accuracy:    {grid.score(X_test, y_test):.3f}')

## 6 — Post-Pruning via ccp_alpha

Grow the tree fully, then sweep `ccp_alpha` (cost complexity) to find the right size.

In [ ]:
tree_full = DecisionTreeClassifier(random_state=42)
path = tree_full.cost_complexity_pruning_path(X_train, y_train)
alphas = path.ccp_alphas
print(f'Number of candidate alphas: {len(alphas)}')
print(f'Range: {alphas[0]:.5f} to {alphas[-1]:.5f}')

In [ ]:
# Try a sample of alphas
rows = []
for a in alphas[::3]:                       # every 3rd value to keep table short
    m = DecisionTreeClassifier(ccp_alpha=a, random_state=42).fit(X_train, y_train)
    rows.append({
        'ccp_alpha': round(a, 5),
        'leaves':    m.get_n_leaves(),
        'train_acc': round(m.score(X_train, y_train), 3),
        'test_acc':  round(m.score(X_test,  y_test),  3)
    })
print(pd.DataFrame(rows))

Smaller `ccp_alpha` → bigger tree (closer to overfit). Larger → smaller, simpler tree. Pick the alpha where test accuracy peaks.

## 7 — Classification Report and Confusion Matrix

In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=target_names))

## 8 — Feature Importance

In [ ]:
importance = pd.DataFrame({
    'feature':    feature_names,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

print(importance.head(10).round(3))

In [ ]:
# Bar chart of top 10 features
top = importance.head(10)
plt.figure(figsize=(10, 5))
plt.barh(top['feature'][::-1], top['importance'][::-1])
plt.xlabel('Importance')
plt.title('Top 10 Feature Importances')
plt.tight_layout()
plt.show()

## 9 — Regression Tree on Diabetes Data

Now switch to regression — predict disease progression score (continuous).

In [ ]:
data_r = load_diabetes()
X_r, y_r = data_r.data, data_r.target
feature_names_r = data_r.feature_names

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_r, y_r, test_size=0.2, random_state=42
)
print('Shape:', X_r.shape)
print('Target range:', y_r.min(), 'to', y_r.max())

In [ ]:
rows = []
for d in [2, 3, 5, 7, 10, None]:
    m = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_train_r, y_train_r)
    pred_train = m.predict(X_train_r)
    pred_test  = m.predict(X_test_r)
    rows.append({
        'max_depth':   d if d is not None else 'unbounded',
        'leaves':      m.get_n_leaves(),
        'train_R2':    round(r2_score(y_train_r, pred_train), 3),
        'test_R2':     round(r2_score(y_test_r,  pred_test),  3),
        'test_MSE':    round(mean_squared_error(y_test_r, pred_test), 1)
    })
print(pd.DataFrame(rows))

Same pattern as classification — unbounded tree has perfect train R² but poor test R². Around `max_depth=3–5` strikes the best balance.

## Summary

- Unbounded tree → memorises training data (overfit)
- Pre-pruning (`max_depth`, `min_samples_*`) → simple way to control complexity
- Post-pruning (`ccp_alpha`) → grows fully, then trims; tune via CV
- `GridSearchCV` → standard way to pick the right balance
- `feature_importances_` → quick interpretability of which features drove the splits
- Same API for both classification (`DecisionTreeClassifier`) and regression (`DecisionTreeRegressor`)